# 05 章 / 第 2 节 · 全参 SFT + Liger Kernel(吞吐 & 显存对照)

## 学习目标
- 用 **Liger Kernel**(LinkedIn 开源)把 Qwen2.5-1.5B 全参 SFT 塞进 12GB 卡
- 理解 Liger 怎么省显存(fused RMSNorm / RoPE / SwiGLU / **fused cross-entropy** —— 不再实例化 logits 大张量)
- 出 **开/关 Liger 的吞吐 + 显存对照表**(面试加分项)
- 与 03 节 Unsloth LoRA 对比:全参效果更高 / 显存更紧

## 对应八股
`llm-interview/llm-ft.md`(全参 vs PEFT)

## ⚠️ 运行要求
**Linux + CUDA GPU 12GB+**;`liger-kernel` 仅 Linux,Windows 跳过本节。


## 1. 概念背景

### 1.1 全参 SFT 为什么显存大

Qwen2.5-1.5B (bf16) 训练显存分布:
- 参数:1.54B × 2B = **3 GB**
- 梯度:同上,**3 GB**
- AdamW 优化器 state(m, v):4 × 1.54B = **6 GB**
- Activations(forward 中间结果):随 batch × seq_len 涨,**5-10 GB**
- **总计 ~17-22 GB**,12GB 卡 OOM

### 1.2 Liger Kernel 怎么解决

**核心 trick:fused cross-entropy**。
传统训练 logits 张量 `(batch × seq × vocab)`,Qwen vocab = 151k,
batch=4 seq=2048 时 logits = 4×2048×151k×bf16 = **4.6 GB** 仅 forward 一步!
Liger 把 cross-entropy 和 final lm_head 融合,**根本不实例化 logits**。

加上 fused RMSNorm/RoPE/SwiGLU,**整体 -60% 显存,+20% 吞吐**(HF 官方测)。

### 1.3 vs Unsloth

- Unsloth:更激进(自带 4bit、自家 ckpt、custom backward),单卡省最多
- Liger:更轻量(只换 forward kernel),HF 生态友好,**多卡 / DeepSpeed / FSDP 都兼容**
- 二选一,不要同时开


## 2. 环境检查


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
from scripts.env_check import check
check(extras=("liger_kernel",))


## 3. 一行启用 Liger(monkey-patch Qwen2 模块)


In [ ]:
# 在 import model 之前调用!
from liger_kernel.transformers import apply_liger_kernel_to_qwen2

apply_liger_kernel_to_qwen2(
    rope = True,
    cross_entropy = False,    # 关 普通 CE,开 fused linear CE
    fused_linear_cross_entropy = True,  # ⭐ 显存大头
    rms_norm = True,
    swiglu = True,
)
print("Liger Kernel 已应用到 Qwen2 架构")


## 4. 加载模型 + 数据


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype = torch.bfloat16,
    device_map = "cuda",
    attn_implementation = "flash_attention_2",  # FA2 配合 Liger 效果最好
)
model.gradient_checkpointing_enable()  # 必开,再省 30% 显存
print(f"模型加载,显存: {torch.cuda.memory_allocated()/1e9:.2f} GB")

raw = load_dataset("shibing624/alpaca-zh", split="train").shuffle(seed=42).select(range(2000))
def fmt(ex):
    msgs = [{"role": "user", "content": ex["instruction"] + (("\n\n" + ex["input"]) if ex.get("input") else "")},
            {"role": "assistant", "content": ex["output"]}]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False)}
ds = raw.map(fmt, remove_columns=raw.column_names)


## 5. SFTTrainer 配置(强制 packing)


In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "../checkpoints/05_sft_liger_full"

cfg = SFTConfig(
    output_dir = OUTPUT_DIR,
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 8,    # 等效 batch=16
    max_steps = 200,
    learning_rate = 1e-5,               # 全参 lr 比 LoRA 低 20x
    warmup_steps = 20,
    lr_scheduler_type = "cosine",
    weight_decay = 0.01,
    optim = "adamw_torch_fused",        # 全参 fused adamw 比 8bit 快
    bf16 = True,
    max_length = 2048,
    packing = True,                     # ⭐ 必开
    dataset_text_field = "text",
    gradient_checkpointing = True,
    logging_steps = 10,
    save_steps = 100,
    save_total_limit = 2,
    report_to = "none",
    seed = 42,
)

trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds, args=cfg)


## 6. 训练 + 显存 / 吞吐统计


In [ ]:
import time

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
stats = trainer.train()
elapsed = time.time() - t0
peak_gb = torch.cuda.max_memory_allocated() / 1e9

n_steps = cfg.max_steps
n_tokens = n_steps * cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps * cfg.max_length
tput_tok_per_s = n_tokens / elapsed

print(f"\n=== 训练完成(Liger 开)===")
print(f"  总耗时:      {elapsed/60:.2f} min")
print(f"  显存峰值:    {peak_gb:.2f} GB")
print(f"  吞吐:        {tput_tok_per_s:.0f} tokens/s")
print(f"  末步 loss:   {stats.training_loss:.4f}")

# 把数字写文件,后面对比用
import json
pathlib.Path("../checkpoints/05_sft_liger_full").mkdir(parents=True, exist_ok=True)
with open("../checkpoints/05_sft_liger_full/liger_on_metrics.json", "w") as f:
    json.dump({"peak_gb": peak_gb, "tput": tput_tok_per_s, "loss": stats.training_loss}, f, indent=2)


## 7. 对照实验(可选,需要重启 kernel 关 Liger 重跑)

把上面的 `apply_liger_kernel_to_qwen2(...)` 全注释掉,重跑训练,记录 `peak_gb` 和 `tput`。
预期数字(4070 12GB, batch=2, seq=2048):

| 配置 | 显存峰值 | 吞吐 | 总耗时 |
|---|---|---|---|
| 关 Liger | OOM 或 11.8 GB(临界) | ~700 tok/s | 6 min |
| 开 Liger | **6.5 GB** | **920 tok/s** | **4.5 min** |

差异主要来自 fused linear cross-entropy 不实例化 logits。

## 8. 踩坑记录

- **`apply_liger_kernel_to_qwen2` 必须在 `from_pretrained` 之前**:monkey-patch 是改类不改实例,模型加载后再 patch 没用
- **`fused_linear_cross_entropy=True` 必须配 `cross_entropy=False`**:同时开两个 CE 实现 LM head 会被装饰两次,报错
- **`gradient_checkpointing` 不能关**:Liger 省 forward 显存,activations 还得靠 ckpt 省;两者叠加才进 12GB
- **`adamw_torch_fused` 比 `adamw_8bit` 快**:全参时 fused 用 fp32 master weights 算得稳,8bit 多一次量化反量化反而慢;PEFT 时反过来
- **多卡 + Liger 用 FSDP 而非 DDP**:Liger 改 forward kernel,DDP 的梯度同步对它透明;FSDP 共享 state 更省

## 9. 延伸阅读

- [Liger Kernel 仓库](https://github.com/linkedin/Liger-Kernel)
- [Liger 论文 `/paper fetch 2410.10989`]
- [TRL packing 文档](https://huggingface.co/docs/trl/sft_trainer#packing-dataset-)
- 本仓:[`03_unsloth_lora.ipynb`](03_unsloth_lora.ipynb)(对比 LoRA 路径)
- [[Slipbox/liger-kernel-how-it-works]] / [[Slipbox/fused-linear-cross-entropy]]
